# Agent Factory

The `factory.py` module builds a compiled LangGraph agent that repeatedly calls a chat model, executes requested tools, applies middleware, and stops when the model no longer requests additional tool calls or structured-output processing.

The public API is centered on `create_agent`. The overloads preserve the type of the agent's structured response, while the concrete implementation builds the state graph, middleware nodes, model and tool nodes, routing edges, persistence configuration, cache integration, and stream transformers.

## Constants

1. `STRUCTURED_OUTPUT_ERROR_TEMPLATE`: Stores the default error message returned to the model when structured-output validation fails and the configured strategy allows a retry.

   * **Definition:**
     ```python
     STRUCTURED_OUTPUT_ERROR_TEMPLATE = (
         "Error: {error}\n"
         " Please fix your mistakes."
     )
     ```

2. `DYNAMIC_TOOL_ERROR_TEMPLATE`: Stores the diagnostic message raised when middleware adds client-side tools that were not registered when the agent was created.

   The message explains that the tool must either be registered through `create_agent`, exposed through `AgentMiddleware.tools`, or handled dynamically through `wrap_tool_call`.

   * **Type:**
     ```python
     DYNAMIC_TOOL_ERROR_TEMPLATE: str
     ```

3. `FALLBACK_MODELS_WITH_STRUCTURED_OUTPUT`: Stores regular-expression patterns for model names assumed to support provider-native structured output when model-profile information is unavailable.

   * **Type:**
     ```python
     FALLBACK_MODELS_WITH_STRUCTURED_OUTPUT: list[str]
     ```

### Functions

1. `create_agent`: Creates and compiles an agent graph that calls a chat model and executes tools in a loop.
   The agent adds the optional system message before model invocation. When the returned `AIMessage` contains tool calls, the graph routes to the tool node, appends the resulting `ToolMessage` values, and returns to the model. The loop finishes when no executable tool calls remain, a structured response is produced, a return-direct tool completes, or middleware redirects execution to the end.
   * **Syntax:**
     ```python
        create_agent(
            model: str | BaseChatModel,
            # LLM used by the agent.
            # Can be a model name or an initialized chat model object.

            tools: Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None = None,
            # Tools the agent is allowed to call.
            # Can include LangChain tools, normal Python functions,
            # or provider-specific tool definitions.

            *,

            system_prompt: str | SystemMessage | None = None,
            # Main instructions that control how the agent should behave.

            middleware: Sequence[AgentMiddleware[StateT_co, ContextT]] = (),
            # Extra logic that can run before, during, or after agent execution.
            # It can modify prompts, tools, model settings, state, or tool results.

            response_format: ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None = None,
            # Defines the required structure of the final response.
            # Use a Pydantic model or schema for structured output.
            # Use None for normal text output.

            state_schema: type[AgentState[ResponseT]] | None = None,
            # Default state already contains conversation messages. custom_state add extra fields, and these fields updates at various stage of agent's lifecycle

            context_schema: type[ContextT] | None = None,
            # Defines fixed runtime information passed when invoking the agent.
            # Examples: user_id, user_role, tenant_id, or database settings.
            # Context is available to tools and middleware but is not normally
            # Changed by the agent or persisted between separate invocations.

            checkpointer: Checkpointer | None = None,
            # Saves and restores state for a particular conversation thread. i.e. Same thread_id = continue the same conversation.
            # Using the same thread_id allows the agent to remember earlier messages and resume interrupted execution.

            store: BaseStore | None = None,
            # Long-term storage shared across multiple conversation threads.
            # Useful for saving user preferences, profile details, or reusable facts.

            interrupt_before: list[str] | None = None,
            # Pauses execution before the specified graph nodes run.
            # Useful when human approval is required before a sensitive action.

            interrupt_after: list[str] | None = None,
            # Pauses execution after the specified graph nodes finish.
            # Useful for reviewing a result before continuing.

            debug: bool = False,
            # Enables detailed execution logs for nodes, state changes, tool calls, and graph routing.

            name: str | None = None,
            # Name assigned to the compiled agent.
            # Used mainly for tracing, debugging, and LangSmith metadata.

            cache: BaseCache[Any] | None = None,
            # Stores reusable execution results to avoid repeating the same model calls or computations.

            transformers: Sequence[TransformerFactory] | None = None
            # Functions that modify or process streamed agent events and output.

       ) -> CompiledStateGraph[
                AgentState[ResponseT],
                # Complete internal state used by the agent during execution.
                # It contains messages and any custom fields defined in state_schema.

                ContextT,
                # Runtime context type defined by context_schema.
                # It contains fixed information such as user_id, role, or configuration.

                InputAgentState,
                # Input structure accepted by agent.invoke().
                # Represents the input-facing subset of that state_schema.

                OutputAgentState[ResponseT]
                # Output structure returned after execution finishes.
                # It contains the final updated state and, when configured, the structured response defined by response_format.
            ] # Returns a compiled and executable LangGraph agent.

        """
        overloads:
        As accroding to response_format. responseT changes
        1. if response_format is None then ResponseT become Any
        2. if `response_format` is a raw then ResponseT become dict[str,Any]
        3. if `response_format` is a response strategy or Python schema type then ResponseT is inferred from that schema
        """
     ```
## Errors
`create_agent` may raise:
1. `AssertionError`: Duplicate middleware names were supplied.
2. `ValueError`: Middleware introduced an unknown client-side tool without a tool-call wrapper.
3. `ValueError`: A dynamically selected `ToolStrategy` refers to a structured-output tool that was not declared in the original response format.
4. `StructuredOutputValidationError`: Provider-native or tool-based structured output failed validation and retry handling was disabled.
5. `MultipleStructuredOutputsError`: More than one structured-output tool was called and retry handling was disabled.
6. `NotImplementedError`: Model-call middleware returned unsupported command fields such as `goto`, `resume`, or `graph`.
7. Provider, tool, middleware, graph-compilation, persistence, and cache exceptions are allowed to propagate.
